# 6.33 - CertCF Adaptive Epsilon Comparison

Compare the original CertCF epsilon rule against the shrink-only adaptive epsilon rule on one compact tabular dataset. The notebook builds two atlases with identical settings except for `adaptive_eps`, then compares build diagnostics and query performance.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from certcf import NearestOppositeClassClearanceStrategy
from counterfactuals.datasets.loaders import CompasDataset
from counterfactuals.methods.certcf import CertCF
from scripts.benchmark import _build_torch_model_from_checkpoint

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)
plt.rcParams.update({'figure.dpi': 120})

## Configuration

In [ ]:
DATASET = 'adult'
SEED = 42
ALPHA = 0.5
CLASSIFICATION_MARGIN = 1.0e-4
DEVICE = 'auto'

# Keep this modest for interactive debugging; set to None for full support.
# K_PER_CLASS = 10000
K_PER_CLASS = 100
N_QUERIES = 200

CERTCF_COMMON = dict(
    norm=1,
    distance_norm=1,
    lirpa_method='backward',
    eps_strategy=NearestOppositeClassClearanceStrategy(alpha=ALPHA),
    batch_size=128,
    default_query_method='nearest_anchor',
    query_k_candidates=5,
    solver_maxiter=500,
    query_parallelism=1,
    k_per_class=K_PER_CLASS,
    subsample_method='random',
    classification_margin=CLASSIFICATION_MARGIN,
    random_seed=SEED,
)

ADAPTIVE_EPS_SHRINK_PARAMS = dict(
    adaptive_eps=True,
    adaptive_eps_shrink_factor=0.5,
    adaptive_eps_max_shrinks=8,
    adaptive_eps_min=1.0e-6,
    adaptive_eps_center_tol=1.0e-6,
    adaptive_eps_binary_search_steps=0,
)

## Load Dataset And Model

In [ ]:
loader = CompasDataset(data_dir=str(ROOT / 'data'), seed=SEED)
loader.load()
x_train, y_train_dataset = loader.get_train()
x_test, y_test = loader.get_test()
spec = loader.spec
cat_slices = list(spec.categorical_slices)

model = _build_torch_model_from_checkpoint(
    checkpoint=str(ROOT / 'checkpoints/compas_classifier/best.ckpt'),
    device=DEVICE,
    dataset_module='compas',
    hidden_dims=[64, 32],
    dropout=0.2,
)

y_train_model = model.predict(x_train).astype(np.int64)
y_test_model = model.predict(x_test).astype(np.int64)

rng = np.random.default_rng(SEED)
query_idx = rng.choice(np.arange(len(x_test)), size=min(N_QUERIES, len(x_test)), replace=False)
x_query = x_test[query_idx]
y_query_orig = y_test_model[query_idx]
y_target = 1 - y_query_orig

print({
    'dataset': DATASET,
    'encoded_dim': int(x_train.shape[1]),
    'n_train': int(len(x_train)),
    'n_test': int(len(x_test)),
    'train_label_source': 'model_prediction',
    'train_dataset_vs_model_agreement': float(np.mean(y_train_dataset == y_train_model)),
    'train_model_counts': dict(zip(*np.unique(y_train_model, return_counts=True))),
    'n_queries': int(len(x_query)),
    'alpha': ALPHA,
    'k_per_class': K_PER_CLASS,
})

## Build Both Atlases

In [ ]:
def atlas_diagnostics(method: CertCF, variant: str, build_time_s: float) -> tuple[dict, pd.DataFrame]:
    atlas = method.atlas
    per_class = []
    for label in atlas.class_labels:
        bd = atlas.bounds[int(label)]
        eps = np.asarray(bd['eps'], dtype=float)
        eps_initial = np.asarray(bd.get('eps_initial', eps), dtype=float)
        shrinks = np.asarray(bd.get('adaptive_eps_n_shrinks', np.zeros_like(eps)), dtype=float)
        binary_steps = np.asarray(bd.get('adaptive_eps_n_binary_steps', np.zeros_like(eps)), dtype=float)
        certified = np.asarray(bd.get('adaptive_eps_center_certified', np.ones_like(eps, dtype=bool)), dtype=bool)
        slack = np.asarray(bd.get('adaptive_eps_center_slack', np.full_like(eps, np.nan)), dtype=float)
        per_class.append(pd.DataFrame({
            'variant': variant,
            'class_label': int(label),
            'eps_initial': eps_initial,
            'eps_final': eps,
            'eps_ratio': np.divide(eps, eps_initial, out=np.ones_like(eps), where=eps_initial > 0),
            'n_shrinks': shrinks,
            'n_binary_steps': binary_steps,
            'n_lirpa_calls': 1.0 + shrinks + binary_steps,
            'center_certified': certified,
            'center_slack': slack,
        }))
    poly_df = pd.concat(per_class, ignore_index=True)
    return {
        'variant': variant,
        'build_time_s': float(build_time_s),
        'n_polytopes': int(len(poly_df)),
        'center_certified_pct': 100.0 * float(poly_df['center_certified'].mean()),
        'shrunk_pct': 100.0 * float((poly_df['n_shrinks'] > 0).mean()),
        'n_shrinks_mean': float(poly_df['n_shrinks'].mean()),
        'binary_steps_mean': float(poly_df['n_binary_steps'].mean()),
        'lirpa_calls_mean': float(poly_df['n_lirpa_calls'].mean()),
        'lirpa_calls_p95': float(poly_df['n_lirpa_calls'].quantile(0.95)),
        'eps_initial_median': float(poly_df['eps_initial'].median()),
        'eps_final_median': float(poly_df['eps_final'].median()),
        'eps_ratio_mean': float(poly_df['eps_ratio'].mean()),
    }, poly_df


def build_certcf_variant(variant: str, variant_params: dict) -> tuple[CertCF, dict, pd.DataFrame]:
    params = dict(CERTCF_COMMON)
    params['ohe_slices'] = cat_slices
    params.update(variant_params)

    method = CertCF(model=model, **params)
    t0 = time.perf_counter()
    method.fit(x_train=x_train, y_train=y_train_model)
    build_time_s = time.perf_counter() - t0
    summary, poly_df = atlas_diagnostics(method, variant=variant, build_time_s=build_time_s)
    return method, summary, poly_df

variants = [
    ('original eps', {'adaptive_eps': False}),
    ('shrink eps', ADAPTIVE_EPS_SHRINK_PARAMS),
]

METHODS = {}
build_rows = []
poly_rows = []
for variant, variant_params in variants:
    print(f'building {variant}')
    method, summary, poly_df = build_certcf_variant(variant, variant_params=variant_params)
    METHODS[variant] = method
    build_rows.append(summary)
    poly_rows.append(poly_df)

BUILD_SUMMARY_DF = pd.DataFrame(build_rows)
POLYTOPE_DIAGNOSTICS_DF = pd.concat(poly_rows, ignore_index=True)
display(BUILD_SUMMARY_DF.round(4))

## Epsilon Shrink Diagnostics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for variant, group in POLYTOPE_DIAGNOSTICS_DF.groupby('variant', sort=False):
    max_shrinks = int(max(1, group['n_shrinks'].max()))
    axes[0].hist(group['n_shrinks'], bins=np.arange(max_shrinks + 2) - 0.5, alpha=0.55, label=variant, density=True)
    axes[1].hist(group['eps_ratio'], bins=30, alpha=0.55, label=variant, density=True)

axes[0].set_title('Number of epsilon halvings')
axes[0].set_xlabel('n shrinks')
axes[0].set_ylabel('density')
axes[1].set_title('Final epsilon / initial epsilon')
axes[1].set_xlabel('eps ratio')
axes[1].set_ylabel('density')
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)
fig.tight_layout();

## LiRPA Repetition Overhead

`n_lirpa_calls = 1 + n_shrinks + n_binary_steps` counts how many LiRPA bound computations are used per atlas center.

In [ ]:
LIRPA_OVERHEAD_DF = (
    POLYTOPE_DIAGNOSTICS_DF
    .groupby('variant', sort=False)
    .agg(
        n_polytopes=('variant', 'size'),
        n_shrinks_mean=('n_shrinks', 'mean'),
        n_shrinks_p95=('n_shrinks', lambda s: float(np.quantile(s, 0.95))),
        n_binary_steps_mean=('n_binary_steps', 'mean'),
        n_binary_steps_p95=('n_binary_steps', lambda s: float(np.quantile(s, 0.95))),
        n_lirpa_calls_mean=('n_lirpa_calls', 'mean'),
        n_lirpa_calls_p95=('n_lirpa_calls', lambda s: float(np.quantile(s, 0.95))),
        shrunk_pct=('n_shrinks', lambda s: 100.0 * float(np.mean(s > 0))),
    )
    .reset_index()
)

display(LIRPA_OVERHEAD_DF.round(4))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(LIRPA_OVERHEAD_DF['variant'], LIRPA_OVERHEAD_DF['n_lirpa_calls_mean'], color=['#4c78a8', '#f58518'])
ax.set_title('Average LiRPA calls per atlas center')
ax.set_ylabel('calls')
ax.grid(axis='y', alpha=0.25)
ax.tick_params(axis='x', rotation=15)
fig.tight_layout();

## Query Comparison

In [ ]:
def query_variant(method: CertCF, variant: str) -> pd.DataFrame:
    rows = []
    t0 = time.perf_counter()
    results = method.generate_batch(x_query, target_class=y_target)
    total_s = time.perf_counter() - t0
    for local_idx, result in enumerate(results):
        x_cf = result.x_cf
        rows.append({
            'variant': variant,
            'query_local_idx': int(local_idx),
            'success': bool(result.success),
            'distance': float(result.distance) if np.isfinite(result.distance) else np.nan,
            'l1_distance': float(np.linalg.norm(x_cf - x_query[local_idx], ord=1)) if x_cf is not None else np.nan,
            'target_class': int(y_target[local_idx]),
            'query_time_s': total_s / max(1, len(results)),
            **{f'meta__{k}': v for k, v in result.metadata.items()},
        })
    return pd.DataFrame(rows)

QUERY_RESULTS_DF = pd.concat(
    [query_variant(method, variant) for variant, method in METHODS.items()],
    ignore_index=True,
)

QUERY_SUMMARY_DF = (
    QUERY_RESULTS_DF
    .groupby('variant', sort=False)
    .agg(
        total=('success', 'size'),
        validity_pct=('success', lambda s: 100.0 * float(np.mean(s))),
        l1_mean=('l1_distance', 'mean'),
        l1_median=('l1_distance', 'median'),
        query_time_s=('query_time_s', 'mean'),
    )
    .reset_index()
)
display(QUERY_SUMMARY_DF.round(4))

## Side-By-Side Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

plot_df = QUERY_SUMMARY_DF.copy()
bar_colors = ['#4c78a8', '#f58518']
axes[0].bar(plot_df['variant'], plot_df['validity_pct'], color=bar_colors[:len(plot_df)])
axes[0].set_title('Validity')
axes[0].set_ylabel('%')
axes[0].set_ylim(0, 105)

axes[1].bar(plot_df['variant'], plot_df['l1_mean'], color=bar_colors[:len(plot_df)])
axes[1].set_title('Mean L1 proximity')
axes[1].set_ylabel('L1')

axes[2].bar(BUILD_SUMMARY_DF['variant'], BUILD_SUMMARY_DF['center_certified_pct'], color=bar_colors[:len(BUILD_SUMMARY_DF)])
axes[2].set_title('Center certified')
axes[2].set_ylabel('% of atlas centers')
axes[2].set_ylim(0, 105)

for ax in axes:
    ax.grid(axis='y', alpha=0.25)
    ax.tick_params(axis='x', rotation=15)
fig.tight_layout();

## Failed Query Inspection

In [ ]:
FAILED_QUERIES_DF = QUERY_RESULTS_DF.loc[~QUERY_RESULTS_DF['success']].copy()
cols = [
    'variant', 'query_local_idx', 'target_class', 'distance', 'l1_distance',
    'meta__n_qp_solved', 'meta__n_candidates_considered', 'meta__n_candidates_pruned_by_bound',
]
existing_cols = [c for c in cols if c in FAILED_QUERIES_DF.columns]
print({'n_failed': int(len(FAILED_QUERIES_DF))})
display(FAILED_QUERIES_DF[existing_cols].head(30))

## Fast 50-Query Performance Check

Use the already-built atlases and compare the two CertCF variants on the same small subset of queries.

In [ ]:
N_FAST_QUERIES = 100
FAST_QUERY_SLICE = np.arange(min(N_FAST_QUERIES, len(x_query)))

x_query_fast = x_query[FAST_QUERY_SLICE]
y_target_fast = y_target[FAST_QUERY_SLICE]


def query_variant_fast(method: CertCF, variant: str) -> pd.DataFrame:
    rows = []
    t0 = time.perf_counter()
    results = method.generate_batch(x_query_fast, target_class=y_target_fast)
    total_s = time.perf_counter() - t0
    for local_idx, result in enumerate(results):
        x_cf = result.x_cf
        rows.append({
            'variant': variant,
            'query_local_idx': int(FAST_QUERY_SLICE[local_idx]),
            'success': bool(result.success),
            'l1_distance': float(np.linalg.norm(x_cf - x_query_fast[local_idx], ord=1)) if x_cf is not None else np.nan,
            'query_time_s': total_s / max(1, len(results)),
            'n_qp_solved': float(result.metadata.get('n_qp_solved', np.nan)),
            'n_candidates_considered': float(result.metadata.get('n_candidates_considered', np.nan)),
        })
    return pd.DataFrame(rows)

FAST_QUERY_RESULTS_DF = pd.concat(
    [query_variant_fast(method, variant) for variant, method in METHODS.items()],
    ignore_index=True,
)

FAST_QUERY_SUMMARY_DF = (
    FAST_QUERY_RESULTS_DF
    .groupby('variant', sort=False)
    .agg(
        total=('success', 'size'),
        validity_pct=('success', lambda s: 100.0 * float(np.mean(s))),
        l1_mean=('l1_distance', 'mean'),
        l1_median=('l1_distance', 'median'),
        query_time_s=('query_time_s', 'mean'),
        n_qp_solved_mean=('n_qp_solved', 'mean'),
        n_candidates_mean=('n_candidates_considered', 'mean'),
    )
    .reset_index()
)

display(FAST_QUERY_SUMMARY_DF.round(4))

FAST_QUERY_WIDE_DF = FAST_QUERY_RESULTS_DF.pivot(index='query_local_idx', columns='variant', values=['success', 'l1_distance'])
comparison_rows = []
strategy_names = [variant for variant, _ in variants]
base_variant = strategy_names[0]
for compare_variant in strategy_names[1:]:
    for q_idx in FAST_QUERY_WIDE_DF.index:
        base_success = bool(FAST_QUERY_WIDE_DF.loc[q_idx, ('success', base_variant)])
        compare_success = bool(FAST_QUERY_WIDE_DF.loc[q_idx, ('success', compare_variant)])
        base_l1 = FAST_QUERY_WIDE_DF.loc[q_idx, ('l1_distance', base_variant)]
        compare_l1 = FAST_QUERY_WIDE_DF.loc[q_idx, ('l1_distance', compare_variant)]
        comparison_rows.append({
            'query_local_idx': int(q_idx),
            'comparison': f'{compare_variant} vs {base_variant}',
            'base_success': base_success,
            'compare_success': compare_success,
            'success_changed': base_success != compare_success,
            'l1_base': float(base_l1) if pd.notna(base_l1) else np.nan,
            'l1_compare': float(compare_l1) if pd.notna(compare_l1) else np.nan,
            'l1_delta_compare_minus_base': float(compare_l1 - base_l1) if pd.notna(base_l1) and pd.notna(compare_l1) else np.nan,
        })

FAST_QUERY_COMPARISON_DF = pd.DataFrame(comparison_rows)
FAST_QUERY_CHANGE_SUMMARY_DF = (
    FAST_QUERY_COMPARISON_DF
    .groupby('comparison', sort=False)
    .agg(
        n_queries=('query_local_idx', 'size'),
        success_changed=('success_changed', 'sum'),
        compare_improved_success=('compare_success', lambda s: int((~FAST_QUERY_COMPARISON_DF.loc[s.index, 'base_success'] & s).sum())),
        compare_lost_success=('compare_success', lambda s: int((FAST_QUERY_COMPARISON_DF.loc[s.index, 'base_success'] & ~s).sum())),
        mean_l1_delta_on_common_successes=('l1_delta_compare_minus_base', 'mean'),
    )
    .reset_index()
)
display(FAST_QUERY_CHANGE_SUMMARY_DF.round(4))
display(FAST_QUERY_COMPARISON_DF.loc[FAST_QUERY_COMPARISON_DF['success_changed']].head(20))